# Social Attraction

## §1. Imports and bot registry

**Edit IPs and uncomment the bots you're using.** For the first multi-bot test, leave just 2 bots active and comment out the rest. Each bot must be running the server with the matching `my_own_color`.


In [ ]:
import random
import requests
import time
import threading
import math
import io
from urllib.parse import urljoin
from IPython.display import display
import ipywidgets as widgets
from PIL import Image as PILImage, ImageDraw
import base64

# ── EDIT THIS ────────────────────────────────────────────────────────
# Comment out bots you're not using right now. Start with just 2 for first test.
bot_registry = [
    {'bot_id': 'bot_1', 'ip_address': '194.47.156.39',  'assigned_color': 'blue'},
    {'bot_id': 'bot_2', 'ip_address': '194.47.156.43', 'assigned_color': 'purple'},
    {'bot_id': 'bot_3', 'ip_address': '194.47.156.201',  'assigned_color': 'orange'},
    {'bot_id': 'bot_4', 'ip_address': '194.47.156.213',  'assigned_color': 'yellow'},
]
bot_http_port = 8080
# ──────────────────────────────────────────────────────────────────────

def base_url_for_bot(record):
    return f"http://{record['ip_address']}:{bot_http_port}"

http_session_by_bot_id = {b['bot_id']: requests.Session() for b in bot_registry}
bot_record_by_id = {b['bot_id']: b for b in bot_registry}
known_bot_ids = [b['bot_id'] for b in bot_registry]
color_to_bot_id = {b['assigned_color']: b['bot_id'] for b in bot_registry}

display_color_hex_for_color = {
    'red':    '#ff0000', 'orange': '#ff8c00', 'yellow': '#ffd000',
    'blue':   '#0064ff', 'purple': '#c800c8',
}

print(f"Registered {len(bot_registry)} bot(s):")
for b in bot_registry:
    print(f"  {b['bot_id']:<8s}  {b['ip_address']:<18s}  color={b['assigned_color']}")


## §2. Parameters

All v5 parameters carry over unchanged. New parameters for peer avoidance are at the bottom.


In [ ]:
control_parameters_default = {
    # ── v5 Approach ─────────────────────────────────────────────
    'target_stopping_distance_meters':            0.30,      #sos 0.05,
    'approach_complete_zone_meters':              0.02,
    'approach_bearing_tolerance':                 0.18,
    'approach_bearing_tolerance_final':           0.12,
    'forward_creep_when_bearing_off':             0.20,

    # ── Frame-fill arrival ──────────────────────────────────────
    'arrived_cube_fills_frame_ratio':             0.15,
    'arrived_cube_fills_frame_hysteresis':        0.05,
    'close_range_fill_ratio':                     0.10,
    'close_range_bearing_tolerance':              0.35,

    # ── Un-arrival ───────────────────────────────────────────────
    'unarrive_distance_threshold_meters':         0.30,     #sos   0.40,
    'unarrive_consecutive_frames':                20,

    # ── Motor budget ────────────────────────────────────────────
    'maximum_motor_speed':                        0.18,
    'motor_stiction_floor':                       0.10,
    'forward_minimum_to_move':                    0.10,
    'turn_deadband_in_local_x':                   0.04,

    # ── Cube tracking gains ─────────────────────────────────────
    'forward_pull_gain_per_meter':                1.5,
    'turn_pull_gain_per_x_normalized':            0.5,

    # ── Arc-not-pivot ───────────────────────────────────────────
    'target_arc_minimum_wheel_ratio':             0.20,
    'turn_to_wheel_diff_scale':                   0.5,

    # ── Search / lock ───────────────────────────────────────────
    'lost_rotation_speed':                        0.10,
    'cube_lock_grace_frames':                     4,

    # ── Loop / network ──────────────────────────────────────────
    'control_loop_period_seconds':                0.10,
    'http_request_timeout_seconds':               0.6,
    'command_watchdog_ttl_ms':                    500,

    # ── Peer repulsion ──────────────────────────────────────────
    'peer_repulsion_range_meters':                1.00,
    'peer_repulsion_weight':                      0.6,
    'peer_repulsion_falloff_exponent':            2.0,
    'peer_critical_distance_meters':              0.30,
    'peer_minimum_useful_distance_meters':        0.05,

    # ── Caution slowdown ────────────────────────────────────────
    'peer_slowdown_range_meters':                 0.80,
    'peer_slowdown_x_normalized_band':            0.40,
    'peer_slowdown_minimum_factor':               0.30,

    # ── Tracker dead-band (fill-scaled) ─────────────────────────
    'tracker_turn_gain':                          0.7,
    'tracker_turn_speed_max':                     0.12,
    'tracker_dead_band_far':                      0.06,
    'tracker_dead_band_mid':                      0.15,
    'tracker_dead_band_close':                    0.30,
    'tracker_fill_threshold_mid':                 0.15,
    'tracker_fill_threshold_close':               0.30,

    # ── Camera ───────────────────────────────────────────────────
    'camera_horizontal_fov_degrees':              62.0,

    # ── Logging ──────────────────────────────────────────────────
    'log_every_n_iterations':                     5,

    # ── Stage 4: Social attractor ────────────────────────────────
    'social_target_required_distance_advantage':  0.25,
    'social_target_in_path_bearing_band':         0.30,
    'social_target_following_distance':           0.30,      #sos  0.55,
    'social_target_approach_speed':               0.22,
    'social_target_turn_gain':                    0.6,
    'social_target_pull_gain_per_meter':          1.0,

    # ── Stage 5: Brain smoothing + leader hysteresis ────────────
    # Smoothing weight on prior value. Higher = smoother but laggier.
    'brain_distance_ema_alpha':                   0.6,
    # Reading older than this is treated as None in leader selection.
    'brain_distance_stale_seconds':               1.5,
    # New leader must beat current by BOTH of these to take over.
    'leader_change_minimum_distance_advantage':   0.25,
    'leader_change_minimum_relative_advantage':   0.20,

    # ── Stage 5.1: Stall detection ──────────────────────────────
    # Mean abs pixel diff (32x32 downsample, 0-255) below this = no motion seen.
    'stall_detection_pixel_threshold':            5.0,
    # Consecutive frames of commanded motion + low pixel diff to declare stall.
    'stall_detection_frames_threshold':           15,
    # Speed of brief reverse nudge after declaring a stall.
    'stall_recovery_reverse_speed':               0.15,
    # Number of frames to apply the reverse nudge.
    'stall_recovery_reverse_frames':              3,
    # ── Stage 5.2: PSO search ───────────────────────────────────
    'pso_inertia_w':          0.12,
    'pso_cognitive_c1':       1.2,
    'pso_social_c2':          1.2,
    'pso_max_velocity_x':     0.15,
    'pso_max_velocity_y':     0.10,
    # Forward component added to search arcs (replaces pure pivot).
    'search_forward_speed':   0.11,
}


control_parameter_overrides_per_bot = {}

def param(bot_id, key):
    ov = control_parameter_overrides_per_bot.get(bot_id, {})
    return ov[key] if key in ov else control_parameters_default[key]

def set_override(bot_id, key, value):
    if bot_id not in control_parameter_overrides_per_bot:
        control_parameter_overrides_per_bot[bot_id] = {}
    control_parameter_overrides_per_bot[bot_id][key] = value

print('Parameters loaded (Stage 5.1 — state reset + follower stop + stall detection).')


## §3. Network helpers

In [ ]:
# Per-bot frame dimensions, populated from /health
frame_width_by_bot_id  = {bid: 3000 for bid in known_bot_ids}
frame_height_by_bot_id = {bid: 1100 for bid in known_bot_ids}


def fetch_state(bot_id):
    '''Stage 5.1: also return frame_difference_value for stall detection.'''
    record = bot_record_by_id[bot_id]
    session = http_session_by_bot_id[bot_id]
    timeout = param(bot_id, 'http_request_timeout_seconds')
    t0 = time.time()
    try:
        r = session.get(urljoin(base_url_for_bot(record), '/state'), timeout=timeout)
        latency = time.time() - t0
        if r.status_code != 200:
            return False, None, [], latency, 0.0
        payload = r.json()
    except requests.RequestException:
        return False, None, [], time.time() - t0, 0.0
    peers = list(payload.get('peers', []))
    frame_diff = float(payload.get('frame_difference_value', 0.0))
    if not payload.get('cube_visible', False):
        return True, None, peers, latency, frame_diff
    cube = {
        'cube_x_normalized':    float(payload['cube_x_normalized']),
        'cube_y_normalized':    float(payload['cube_y_normalized']),
        'cube_distance_meters': float(payload['cube_distance_meters']),
    }
    bbox = payload.get('cube_bounding_box')
    if bbox is not None and len(bbox) >= 4:
        cube['cube_bounding_box'] = tuple(int(v) for v in bbox[:4])
    return True, cube, peers, latency, frame_diff


def send_command(bot_id, left, right, ttl_ms=None):
    record = bot_record_by_id[bot_id]
    session = http_session_by_bot_id[bot_id]
    timeout = param(bot_id, 'http_request_timeout_seconds')
    if ttl_ms is None:
        ttl_ms = param(bot_id, 'command_watchdog_ttl_ms')
    try:
        r = session.post(
            urljoin(base_url_for_bot(record), '/command'),
            json={'left_motor_speed': float(left),
                  'right_motor_speed': float(right),
                  'command_ttl_milliseconds': int(ttl_ms)},
            timeout=timeout)
        return r.status_code == 200
    except requests.RequestException:
        return False


def stop_bot(bot_id):
    return send_command(bot_id, 0.0, 0.0, ttl_ms=100)


def stop_all_bots():
    for bid in known_bot_ids:
        stop_bot(bid)


def fetch_health(bot_id):
    record = bot_record_by_id[bot_id]
    session = http_session_by_bot_id[bot_id]
    try:
        r = session.get(urljoin(base_url_for_bot(record), '/health'), timeout=1.5)
        return r.json() if r.status_code == 200 else None
    except requests.RequestException:
        return None

print('Network helpers ready.')


## §4. Connection self-test

In [ ]:
def run_self_test():
    print(f'Self-testing {len(bot_registry)} bot(s)...')
    print()
    any_bad = False
    for record in bot_registry:
        bid = record['bot_id']
        h = fetch_health(bid)
        if h is None:
            print(f"  {bid:<8s} ❌ unreachable")
            any_bad = True
            continue
        rate = h.get('detection_loop_hz', 0)
        col  = h.get('bot_own_color')
        fw   = h.get('cropped_frame_width')
        fh   = h.get('cropped_frame_height')
        if fw is not None:
            frame_width_by_bot_id[bid]  = int(fw)
        if fh is not None:
            frame_height_by_bot_id[bid] = int(fh)
        color_match = '✓' if col == record['assigned_color'] else f"⚠ bot says '{col}'"
        rate_flag = '✓' if rate >= 5.0 else ('⚠ marginal' if rate >= 3.0 else '❌ too slow')
        print(f"  {bid:<8s}  rate={rate:4.1f}Hz {rate_flag}  color={col} {color_match}  frame={fw}×{fh}")
        if rate < 3.0 or col != record['assigned_color']:
            any_bad = True
    print()
    print('✓ All good.' if not any_bad else '⚠ Fix issues above before continuing.')

run_self_test()


## §5. Controller — Stage 5.2

Two surgical changes on top of Stage 4:

### Brain state changes

Each bot's cube distance in the brain is now a smoothed (EMA) value with a timestamp. The publish function does the smoothing. The leader-selection function rejects readings older than `brain_distance_stale_seconds`.

### Leader change is gated

`brain_recompute_and_get_leader` now keeps the current leader unless a challenger is both **0.25 m closer AND 20% closer**. Stops the ping-pong observed in Stage 4 logs.

Per-bot smoothed-distance state lives in `brain_state['cube_distance_smoothed_by_bot_id']` and `brain_state['cube_distance_timestamp_by_bot_id']`. Raw distances are still used for the bot's own local decisions (arrival check, approach gains).

Everything else — social attractor, tracker dead-band, arrival logic, leader path, follower path — is byte-identical to Stage 4.

### Stage 5.2 additions (PSO-guided arc search)

Two pivot blocks replaced with forward arcs steered by PSO velocity:
- Each cube sighting updates the bot's personal best (bearing + distance) and the swarm global best.
- When searching, each bot computes a PSO velocity step toward its personal best and the global best bearing.
- Instead of spinning in place, the bot arcs forward — translating while scanning.
- `pso_vx > 0` → arc right; `pso_vx < 0` → arc left; falls back to `last_seen_x` if PSO is flat.
- New parameters: `pso_inertia_w`, `pso_cognitive_c1`, `pso_social_c2`, `pso_max_velocity_x/y`, `search_forward_speed`.


In [ ]:
# ──────────────────────────────────────────────────────────────────
# Brain shared state (unchanged from Stage 3.2)
# ──────────────────────────────────────────────────────────────────
brain_lock = threading.Lock()
brain_state = {
    # RAW most-recent reading per bot (kept for backwards compat / debugging).
    'cube_distance_by_bot_id':            {bid: None for bid in known_bot_ids},
    # Stage 5: smoothed reading + timestamp, used by leader selection.
    'cube_distance_smoothed_by_bot_id':   {bid: None for bid in known_bot_ids},
    'cube_distance_timestamp_by_bot_id':  {bid: 0.0  for bid in known_bot_ids},
    'arrived_bot_ids':                    set(),
    'current_leader_bot_id':              None,
    'last_leader_change_time':            0.0,
    # Stage 5.2: PSO per-bot velocity and personal best (bearing + distance).
    'pso_vel_x_by_bot_id':    {bid: 0.0    for bid in known_bot_ids},
    'pso_vel_y_by_bot_id':    {bid: 0.0    for bid in known_bot_ids},
    'pso_best_x_by_bot_id':   {bid: 0.0    for bid in known_bot_ids},
    'pso_best_dist_by_bot_id':{bid: 9999.0 for bid in known_bot_ids},
    # PSO swarm global best.
    'pso_global_best_x':      0.0,
    'pso_global_best_dist':   9999.0,
}

def brain_publish_cube_distance(bot_id, distance_or_none):
    '''Stage 5: maintain a smoothed estimate + timestamp.

    The raw reading is still recorded for debugging. Leader selection uses
    the smoothed value (and rejects it if stale).
    '''
    alpha = param(bot_id, 'brain_distance_ema_alpha')
    now = time.time()
    with brain_lock:
        brain_state['cube_distance_by_bot_id'][bot_id] = distance_or_none
        if distance_or_none is None:
            # Don't decay or update smoothed — keep last value but mark timestamp.
            # The leader-selection staleness check will drop it.
            brain_state['cube_distance_timestamp_by_bot_id'][bot_id] = now
            return
        prior = brain_state['cube_distance_smoothed_by_bot_id'].get(bot_id)
        if prior is None:
            new_smoothed = float(distance_or_none)
        else:
            new_smoothed = alpha * prior + (1.0 - alpha) * float(distance_or_none)
        brain_state['cube_distance_smoothed_by_bot_id'][bot_id]  = new_smoothed
        brain_state['cube_distance_timestamp_by_bot_id'][bot_id] = now

def brain_mark_arrived(bot_id):
    with brain_lock:
        brain_state['arrived_bot_ids'].add(bot_id)

def brain_mark_unarrived(bot_id):
    with brain_lock:
        brain_state['arrived_bot_ids'].discard(bot_id)

def brain_recompute_and_get_leader():
    '''Stage 5: use SMOOTHED distances; reject stale; hysteresis on switch.'''
    stale_threshold = control_parameters_default['brain_distance_stale_seconds']
    abs_advantage   = control_parameters_default['leader_change_minimum_distance_advantage']
    rel_advantage   = control_parameters_default['leader_change_minimum_relative_advantage']
    now = time.time()
    with brain_lock:
        candidates = []
        arrived = brain_state['arrived_bot_ids']
        smoothed_map = brain_state['cube_distance_smoothed_by_bot_id']
        ts_map       = brain_state['cube_distance_timestamp_by_bot_id']
        for bid, dist in smoothed_map.items():
            if dist is None:
                continue
            if bid in arrived:
                continue
            if (now - ts_map.get(bid, 0.0)) > stale_threshold:
                continue
            candidates.append((dist, bid))
        if not candidates:
            brain_state['current_leader_bot_id'] = None
            return None
        candidates.sort()
        best_dist, best_bid = candidates[0]
        current_leader = brain_state['current_leader_bot_id']

        # Decide whether to switch.
        should_switch = False
        switch_reason = None
        if current_leader is None:
            should_switch = True
            switch_reason = 'no_prior_leader'
        elif current_leader == best_bid:
            should_switch = False
        elif current_leader in arrived:
            should_switch = True
            switch_reason = 'prior_leader_arrived'
        else:
            current_dist = smoothed_map.get(current_leader)
            if current_dist is None:
                should_switch = True
                switch_reason = 'prior_leader_dist_lost'
            elif (now - ts_map.get(current_leader, 0.0)) > stale_threshold:
                should_switch = True
                switch_reason = 'prior_leader_stale'
            else:
                gap_abs = current_dist - best_dist
                gap_rel = gap_abs / max(current_dist, 1e-6)
                if gap_abs >= abs_advantage and gap_rel >= rel_advantage:
                    should_switch = True
                    switch_reason = f'beat_by {gap_abs:.2f}m ({gap_rel*100:.0f}%)'

        if should_switch and current_leader != best_bid:
            brain_state['current_leader_bot_id'] = best_bid
            brain_state['last_leader_change_time'] = now
            ranking_str = ', '.join(f"{bid}={d:.2f}m" for d, bid in candidates)
            arrived_str = ', '.join(sorted(arrived)) if arrived else 'none'
            print(f"  [brain] LEADER -> {best_bid} (cube_d={best_dist:.2f}m, {switch_reason})")
            print(f"          candidates: {ranking_str}")
            if arrived:
                print(f"          arrived (excluded): {arrived_str}")
        return brain_state['current_leader_bot_id']

def brain_get_status_snapshot():
    '''Stage 5: 'distances' returns SMOOTHED values (what social attractor reads).'''
    with brain_lock:
        return {
            'leader':    brain_state['current_leader_bot_id'],
            'arrived':   set(brain_state['arrived_bot_ids']),
            'distances': dict(brain_state['cube_distance_smoothed_by_bot_id']),
        }



# ──────────────────────────────────────────────────────────────────
# Stage 5.2 — PSO brain functions
# ──────────────────────────────────────────────────────────────────
def brain_publish_pso_sighting(bot_id, x_normalized, distance_meters):
    """Update this bot's personal best and the swarm global best.
    Called every frame the cube is visible.
    """
    with brain_lock:
        if distance_meters < brain_state['pso_best_dist_by_bot_id'][bot_id]:
            brain_state['pso_best_x_by_bot_id'][bot_id]    = float(x_normalized)
            brain_state['pso_best_dist_by_bot_id'][bot_id] = float(distance_meters)
        # Recompute global best across all bots.
        best_bot = min(known_bot_ids,
                       key=lambda b: brain_state['pso_best_dist_by_bot_id'][b])
        brain_state['pso_global_best_x']    = brain_state['pso_best_x_by_bot_id'][best_bot]
        brain_state['pso_global_best_dist'] = brain_state['pso_best_dist_by_bot_id'][best_bot]


def brain_compute_pso_velocity(bot_id, current_x, current_dist):
    """Run one PSO velocity step for this bot and return (vx, vy).
    current_x    — bot's current bearing to cube (-1..+1), or last-known.
    current_dist — bot's current distance to cube (m), or 9999 if unseen.
    """
    W      = control_parameters_default['pso_inertia_w']
    C1     = control_parameters_default['pso_cognitive_c1']
    C2     = control_parameters_default['pso_social_c2']
    max_vx = control_parameters_default['pso_max_velocity_x']
    max_vy = control_parameters_default['pso_max_velocity_y']
    with brain_lock:
        vx = brain_state['pso_vel_x_by_bot_id'][bot_id]
        vy = brain_state['pso_vel_y_by_bot_id'][bot_id]
        px = brain_state['pso_best_x_by_bot_id'][bot_id]
        pd = brain_state['pso_best_dist_by_bot_id'][bot_id]
        gx = brain_state['pso_global_best_x']
        gd = brain_state['pso_global_best_dist']
        r1x, r1y = random.random(), random.random()
        r2x, r2y = random.random(), random.random()
        # Bearing velocity: pull toward personal-best and global-best bearing.
        new_vx = (W * vx
                  + C1 * r1x * (px - current_x)
                  + C2 * r2x * (gx - current_x))
        # Distance velocity: pull toward shorter distance.
        new_vy = (W * vy
                  + C1 * r1y * (pd - current_dist)
                  + C2 * r2y * (gd - current_dist))
        new_vx = max(-max_vx, min(max_vx, new_vx))
        new_vy = max(-max_vy, min(max_vy, new_vy))
        brain_state['pso_vel_x_by_bot_id'][bot_id] = new_vx
        brain_state['pso_vel_y_by_bot_id'][bot_id] = new_vy
        return new_vx, new_vy

def brain_reset():
    with brain_lock:
        for bid in known_bot_ids:
            brain_state['cube_distance_by_bot_id'][bid] = None
            brain_state['cube_distance_smoothed_by_bot_id'][bid] = None
            brain_state['cube_distance_timestamp_by_bot_id'][bid] = 0.0
        brain_state['arrived_bot_ids'] = set()
        brain_state['current_leader_bot_id'] = None
        brain_state['last_leader_change_time'] = 0.0
        # Stage 5.2: reset PSO state.
        for bid in known_bot_ids:
            brain_state['pso_vel_x_by_bot_id'][bid]      = 0.0
            brain_state['pso_vel_y_by_bot_id'][bid]      = 0.0
            brain_state['pso_best_x_by_bot_id'][bid]     = 0.0
            brain_state['pso_best_dist_by_bot_id'][bid]  = 9999.0
        brain_state['pso_global_best_x']    = 0.0
        brain_state['pso_global_best_dist'] = 9999.0
    print('Brain reset (Stage 5.2: PSO state cleared too).')


# ──────────────────────────────────────────────────────────────────
# Per-bot state
# ──────────────────────────────────────────────────────────────────
def _default_bot_state():
    return {
        'last_seen_cube_x_normalized': 0.0,
        'consecutive_missed_frames':   0,
        'previous_cube_distance':      None,
        'last_motor_command_left':     0.0,
        'last_motor_command_right':    0.0,
        'arrived_by_fill_latched':     False,
        'frames_cube_departed_count':  0,
        'frames_cube_missing_count':   0,
        # Stage 5.1: stall-detection counters.
        'stall_low_motion_frames':     0,
        'stall_recovery_frames_left':  0,
    }

bot_state = {bid: _default_bot_state() for bid in known_bot_ids}


def reset_bot_states_for_new_run():
    '''Stage 5.1: wipe per-bot controller state. Called by tracking-start.'''
    for bid in known_bot_ids:
        bot_state[bid] = _default_bot_state()
    print('Per-bot controller state reset (Stage 5.1).')


# ──────────────────────────────────────────────────────────────────
# Geometry / motor helpers (unchanged)
# ──────────────────────────────────────────────────────────────────
def compute_adaptive_arc_ratio(bot_id, forward_target_motor_units, turn_intensity):
    target_ratio = (1.0 - (1.0 - param(bot_id, 'target_arc_minimum_wheel_ratio'))
                    * min(1.0, abs(turn_intensity)))
    if abs(forward_target_motor_units) > 1e-6:
        stiction = param(bot_id, 'motor_stiction_floor')
        minimum_safe_ratio = stiction / abs(forward_target_motor_units)
        target_ratio = max(target_ratio, minimum_safe_ratio)
    return min(target_ratio, 1.0)


def arc_wheel_mapping(bot_id, forward_target, turn_intensity):
    max_speed = param(bot_id, 'maximum_motor_speed')
    stiction  = param(bot_id, 'motor_stiction_floor')
    f, t = forward_target, turn_intensity
    if abs(f) >= stiction:
        slower_ratio = compute_adaptive_arc_ratio(bot_id, f, t)
        if t > 0:
            left, right = f, f * slower_ratio
        elif t < 0:
            left, right = f * slower_ratio, f
        else:
            left = right = f
    elif abs(t) >= 0.05:
        rot = max(stiction, abs(t) * max_speed)
        left, right = (+rot, -rot) if t > 0 else (-rot, +rot)
        slower_ratio = 0.0
    else:
        left = right = 0.0
        slower_ratio = 1.0
    left  = max(-max_speed, min(max_speed, left))
    right = max(-max_speed, min(max_speed, right))
    return left, right, slower_ratio


def compute_frame_fill_ratio(bot_id, cube_detection):
    bbox = cube_detection.get('cube_bounding_box') if cube_detection else None
    if bbox is None or len(bbox) < 4:
        return None
    return bbox[2] / max(1, frame_width_by_bot_id[bot_id])


def bearing_to_local_vector(x_normalized, magnitude, fov_degrees):
    half_fov = math.radians(fov_degrees) / 2.0
    bearing_rad = x_normalized * half_fov
    return (magnitude * math.sin(bearing_rad), magnitude * math.cos(bearing_rad))


def compute_peer_repulsion_vector(bot_id, peer_x_normalized, peer_distance_meters):
    range_m = param(bot_id, 'peer_repulsion_range_meters')
    min_useful = param(bot_id, 'peer_minimum_useful_distance_meters')
    if peer_distance_meters >= range_m or peer_distance_meters < min_useful:
        return (0.0, 0.0)
    falloff = param(bot_id, 'peer_repulsion_falloff_exponent')
    weight  = param(bot_id, 'peer_repulsion_weight')
    fov     = param(bot_id, 'camera_horizontal_fov_degrees')
    closeness = max(0.0, (range_m - peer_distance_meters) / range_m)
    magnitude = (closeness ** falloff) * weight
    px, py = bearing_to_local_vector(peer_x_normalized, 1.0, fov)
    return (-magnitude * px, -magnitude * py)


# ── Stage 4 change: aggregate_peer_repulsion gains exclude_color ─────
def aggregate_peer_repulsion(bot_id, peers_list, exclude_color=None):
    """Repulsion + critical-stop summary across visible peers.

    Stage 4: pass exclude_color when computing repulsion for a follower
    that is intentionally driving toward `exclude_color`. The excluded
    peer contributes nothing to repulsion, critical-stop, or slowdown.
    """
    own_color = bot_record_by_id[bot_id]['assigned_color']
    crit_dist = param(bot_id, 'peer_critical_distance_meters')
    min_useful = param(bot_id, 'peer_minimum_useful_distance_meters')
    sum_vx = sum_vy = 0.0
    critical_peer = None
    visible_summary = []
    relevant_peers = []
    for p in peers_list:
        color = p.get('peer_color')
        if color is None or color == own_color:
            continue
        if exclude_color is not None and color == exclude_color:
            continue
        d = float(p.get('peer_distance_meters', 999.0))
        x = float(p.get('peer_x_normalized', 0.0))
        if d < min_useful or d > 5.0:
            continue
        vx, vy = compute_peer_repulsion_vector(bot_id, x, d)
        sum_vx += vx
        sum_vy += vy
        visible_summary.append(f"{color}@{d:.1f}m")
        relevant_peers.append((color, x, d))
        if d < crit_dist:
            if critical_peer is None or d < critical_peer['d']:
                critical_peer = {'color': color, 'd': d, 'x': x}
    return sum_vx, sum_vy, critical_peer, visible_summary, relevant_peers


def compute_caution_slowdown_factor(bot_id, relevant_peers):
    slowdown_range = param(bot_id, 'peer_slowdown_range_meters')
    bearing_band   = param(bot_id, 'peer_slowdown_x_normalized_band')
    min_factor     = param(bot_id, 'peer_slowdown_minimum_factor')
    factor = 1.0
    contributors = []
    for (color, x, d) in relevant_peers:
        if d >= slowdown_range:
            continue
        if abs(x) >= bearing_band:
            continue
        proximity = (slowdown_range - d) / slowdown_range
        this_factor = 1.0 - proximity * (1.0 - min_factor)
        factor *= this_factor
        contributors.append((color, d, this_factor))
    factor = max(min_factor, min(1.0, factor))
    return factor, contributors


def compute_tracker_dead_band(bot_id, fill_ratio):
    if fill_ratio is None:
        return param(bot_id, 'tracker_dead_band_far')
    close_thresh = param(bot_id, 'tracker_fill_threshold_close')
    mid_thresh   = param(bot_id, 'tracker_fill_threshold_mid')
    if fill_ratio >= close_thresh:
        return param(bot_id, 'tracker_dead_band_close')
    elif fill_ratio >= mid_thresh:
        return param(bot_id, 'tracker_dead_band_mid')
    else:
        return param(bot_id, 'tracker_dead_band_far')


def compute_tracker_rotation(bot_id, cube_x_normalized, fill_ratio):
    dead_band  = compute_tracker_dead_band(bot_id, fill_ratio)
    turn_gain  = param(bot_id, 'tracker_turn_gain')
    turn_max   = param(bot_id, 'tracker_turn_speed_max')
    stiction   = param(bot_id, 'motor_stiction_floor')
    if abs(cube_x_normalized) < dead_band:
        return 0.0, 0.0
    desired_turn = turn_gain * cube_x_normalized
    abs_turn = max(stiction, min(turn_max, abs(desired_turn) * turn_max))
    if cube_x_normalized > 0:
        return +abs_turn, -abs_turn
    else:
        return -abs_turn, +abs_turn


def update_unarrival_counters_and_maybe_unarrive(bot_id, cube_detection):
    st = bot_state[bot_id]
    if not st['arrived_by_fill_latched']:
        st['frames_cube_departed_count'] = 0
        st['frames_cube_missing_count'] = 0
        return False
    threshold_d = param(bot_id, 'unarrive_distance_threshold_meters')
    threshold_n = param(bot_id, 'unarrive_consecutive_frames')
    if cube_detection is None:
        st['frames_cube_missing_count'] += 1
    else:
        d = cube_detection['cube_distance_meters']
        st['frames_cube_missing_count'] = 0
        if d > threshold_d:
            st['frames_cube_departed_count'] += 1
        else:
            st['frames_cube_departed_count'] = 0
    if st['frames_cube_departed_count'] >= threshold_n:
        print(f"  [{bot_id}] UNARRIVED (cube departed: "
              f"d>{threshold_d}m for {st['frames_cube_departed_count']} frames)")
        st['arrived_by_fill_latched'] = False
        st['frames_cube_departed_count'] = 0
        st['frames_cube_missing_count'] = 0
        brain_mark_unarrived(bot_id)
        return True
    if st['frames_cube_missing_count'] >= threshold_n:
        print(f"  [{bot_id}] UNARRIVED (cube missing for {st['frames_cube_missing_count']} frames)")
        st['arrived_by_fill_latched'] = False
        st['frames_cube_departed_count'] = 0
        st['frames_cube_missing_count'] = 0
        brain_mark_unarrived(bot_id)
        return True
    return False


# ──────────────────────────────────────────────────────────────────
# Stage 4 NEW — social attractor
# ──────────────────────────────────────────────────────────────────
def find_social_target_peer(bot_id, peers_list, cube_detection):
    """Pick a visible peer to drive toward as social target, or None.

    Only activates when this bot CANNOT see the cube itself.
    Direct observation always beats social guidance.

    Selection priority: arrived peers first, then smallest peer cube distance.
    """
    if not peers_list:
        return None
    # If this bot can see the cube directly, go to it — never follow a peer.
    if cube_detection is not None:
        return None

    advantage_threshold = param(bot_id, 'social_target_required_distance_advantage')
    in_path_band        = param(bot_id, 'social_target_in_path_bearing_band')
    min_useful          = param(bot_id, 'peer_minimum_useful_distance_meters')

    my_cube_d = cube_detection['cube_distance_meters'] if cube_detection is not None else None
    my_cube_x = cube_detection['cube_x_normalized']    if cube_detection is not None else None

    brain_snap = brain_get_status_snapshot()
    cube_distances_by_id = brain_snap['distances']
    arrived_set          = brain_snap['arrived']

    own_color = bot_record_by_id[bot_id]['assigned_color']
    candidates = []
    for p in peers_list:
        color = p.get('peer_color')
        if color is None or color == own_color:
            continue
        peer_bot_id = color_to_bot_id.get(color)
        if peer_bot_id is None:
            continue
        peer_cube_d = cube_distances_by_id.get(peer_bot_id)
        if peer_cube_d is None:
            continue  # peer has no cube view to point us to
        peer_x = float(p['peer_x_normalized'])
        peer_d = float(p['peer_distance_meters'])
        if peer_d < min_useful:
            continue
        is_arrived = peer_bot_id in arrived_set

        reason = None
        if my_cube_d is None:
            reason = 'blind_following_seer'
        elif is_arrived and my_cube_x is not None and abs(peer_x - my_cube_x) < in_path_band:
            reason = 'arrived_in_path'
        elif (peer_cube_d + advantage_threshold) < my_cube_d:
            reason = 'significant_advantage'

        if reason is None:
            continue

        candidates.append({
            'peer_color':           color,
            'peer_x_normalized':    peer_x,
            'peer_distance_meters': peer_d,
            'peer_bot_id':          peer_bot_id,
            'peer_is_arrived':      is_arrived,
            'peer_cube_distance':   peer_cube_d,
            'reason':               reason,
        })

    if not candidates:
        return None

    # Arrived peers sort first (False < True; we want arrived first).
    candidates.sort(key=lambda c: (not c['peer_is_arrived'], c['peer_cube_distance']))
    return candidates[0]


def compute_follow_peer_command(bot_id, social_target, peers_list):
    """Drive toward the social-target peer; stop at following distance.

    Repulsion is computed EXCLUDING the social target. Other peers
    still push us away.
    """
    follow_d       = param(bot_id, 'social_target_following_distance')
    approach_speed = param(bot_id, 'social_target_approach_speed')
    turn_gain      = param(bot_id, 'social_target_turn_gain')
    stiction       = param(bot_id, 'motor_stiction_floor')
    pull_gain      = param(bot_id, 'social_target_pull_gain_per_meter')
    max_speed      = param(bot_id, 'maximum_motor_speed')

    peer_d = social_target['peer_distance_meters']
    peer_x = social_target['peer_x_normalized']

    # Repulsion / slowdown from OTHER peers (excluding the social target).
    rep_vx, rep_vy, _, _, relevant_peers = aggregate_peer_repulsion(
        bot_id, peers_list, exclude_color=social_target['peer_color'])
    slowdown_factor, _ = compute_caution_slowdown_factor(bot_id, relevant_peers)

    # Are we close enough? Stop.
    if peer_d <= follow_d:
        return 0.0, 0.0, {
            'forward_target':  0.0,
            'turn_intensity':  0.0,
            'slowdown_factor': slowdown_factor,
            'arc_ratio':       1.0,
            'stop_reason':     'at_follow_distance',
        }

    # Forward speed proportional to distance error (bounded by approach_speed).
    distance_error = peer_d - follow_d
    forward_target = max(stiction, min(approach_speed, distance_error * pull_gain))
    forward_target = forward_target * slowdown_factor
    if forward_target > 0 and forward_target < stiction:
        forward_target = stiction

    # Turn toward the peer.
    turn_intensity = max(-1.0, min(1.0, turn_gain * peer_x))

    combined_vy = max(-max_speed, min(max_speed, forward_target + rep_vy))
    combined_vx = max(-1.0,        min(1.0,        turn_intensity + rep_vx))

    left, right, slower_ratio = arc_wheel_mapping(bot_id, combined_vy, combined_vx)
    return left, right, {
        'forward_target':  combined_vy,
        'turn_intensity':  combined_vx,
        'slowdown_factor': slowdown_factor,
        'arc_ratio':       slower_ratio,
        'stop_reason':     None,
    }


# ──────────────────────────────────────────────────────────────────
# Main controller
# ──────────────────────────────────────────────────────────────────
def compute_motor_commands(bot_id, cube_detection, peers_list, frame_diff_value=0.0):
    """Stage 5.1: stall detection wraps the Stage 4 social-attractor controller."""
    st = bot_state[bot_id]

    # Publish cube distance + PSO sighting when cube is visible.
    if cube_detection is not None:
        brain_publish_cube_distance(bot_id, cube_detection['cube_distance_meters'])
        brain_publish_pso_sighting(          # Stage 5.2
            bot_id,
            cube_detection['cube_x_normalized'],
            cube_detection['cube_distance_meters'])
    else:
        brain_publish_cube_distance(bot_id, None)

    # ── Stage 5.1: stall recovery — runs BEFORE anything else ────
    # If we're in the middle of a reverse-nudge, keep nudging.
    if st['stall_recovery_frames_left'] > 0:
        st['stall_recovery_frames_left'] -= 1
        rev = 0.0  # stop, not reverse — reversing is unsafe near target/other bots
        st['last_motor_command_left']  = rev
        st['last_motor_command_right'] = rev
        return rev, rev, 'stall_recovery', {
            'lock_state': 'STALL_RECOVER', 'cube_visible': cube_detection is not None,
            'cube_d': cube_detection['cube_distance_meters'] if cube_detection else None,
            'cube_x': cube_detection['cube_x_normalized']    if cube_detection else None,
            'fill_ratio': compute_frame_fill_ratio(bot_id, cube_detection) if cube_detection else None,
            'peers_summary': [p.get('peer_color', '?') for p in peers_list],
            'slowdown_factor': 1.0, 'role': 'STALL_RECOVER',
            'frames_left': st['stall_recovery_frames_left'],
            'left': rev, 'right': rev,
        }

    # Stall detection: were we commanding motion last frame? Did the image change?
    pixel_threshold = param(bot_id, 'stall_detection_pixel_threshold')
    frames_threshold = param(bot_id, 'stall_detection_frames_threshold')
    last_l = st['last_motor_command_left']
    last_r = st['last_motor_command_right']
    was_commanding_motion = (abs(last_l) > 0.05) or (abs(last_r) > 0.05)

    # Only accumulate stall frames while genuinely stuck searching (cube not visible).
    if was_commanding_motion and frame_diff_value < pixel_threshold and cube_detection is None:
        st['stall_low_motion_frames'] += 1
    else:
        st['stall_low_motion_frames'] = 0

    if st['stall_low_motion_frames'] >= frames_threshold:
        # Stall confirmed. Start recovery (brief reverse).
        st['stall_low_motion_frames'] = 0
        st['stall_recovery_frames_left'] = param(bot_id, 'stall_recovery_reverse_frames')
        rev = 0.0  # stop, not reverse
        st['last_motor_command_left']  = rev
        st['last_motor_command_right'] = rev
        print(f"  [{bot_id}] STALL detected after {frames_threshold} frames "
              f"(frame_diff={frame_diff_value:.1f}, motors were L={last_l:+.2f} R={last_r:+.2f}). "
              f"Stopping for {param(bot_id, 'stall_recovery_reverse_frames')} frames.")
        return rev, rev, 'stall_detected', {
            'lock_state': 'STALL', 'cube_visible': cube_detection is not None,
            'cube_d': cube_detection['cube_distance_meters'] if cube_detection else None,
            'cube_x': cube_detection['cube_x_normalized']    if cube_detection else None,
            'fill_ratio': compute_frame_fill_ratio(bot_id, cube_detection) if cube_detection else None,
            'peers_summary': [p.get('peer_color', '?') for p in peers_list],
            'slowdown_factor': 1.0, 'role': 'STALL_RECOVER',
            'frame_diff': frame_diff_value,
            'left': rev, 'right': rev,
        }

    # Full peer repulsion across all visible peers (used for critical-stop check + non-following paths).
    peer_rep_vx, peer_rep_vy, critical_peer, peer_summary, relevant_peers = aggregate_peer_repulsion(
        bot_id, peers_list)
    slowdown_factor, _ = compute_caution_slowdown_factor(bot_id, relevant_peers)

    # 1. Hard safety: any peer at critical distance → stop immediately.
    if critical_peer is not None:
        st['last_motor_command_left'] = st['last_motor_command_right'] = 0.0
        return 0.0, 0.0, 'peer_hard_stop', {
            'lock_state': 'PEER_HOLD', 'cube_visible': cube_detection is not None,
            'critical_peer': critical_peer, 'peers_summary': peer_summary,
            'slowdown_factor': slowdown_factor,
            'role': 'unknown', 'left': 0.0, 'right': 0.0,
        }

    # 2. Already arrived? Sit still. Check un-arrival counters.
    if st['arrived_by_fill_latched']:
        unarrived_now = update_unarrival_counters_and_maybe_unarrive(bot_id, cube_detection)
        if not unarrived_now:
            st['last_motor_command_left'] = st['last_motor_command_right'] = 0.0
            if cube_detection is not None:
                return 0.0, 0.0, 'arrived', {
                    'lock_state': 'ARRIVED', 'cube_visible': True,
                    'cube_d': cube_detection['cube_distance_meters'],
                    'cube_x': cube_detection['cube_x_normalized'],
                    'fill_ratio': compute_frame_fill_ratio(bot_id, cube_detection),
                    'peers_summary': peer_summary, 'slowdown_factor': slowdown_factor,
                    'role': 'ARRIVED',
                    'frames_departed': st['frames_cube_departed_count'],
                    'frames_missing':  st['frames_cube_missing_count'],
                    'left': 0.0, 'right': 0.0,
                }
            else:
                return 0.0, 0.0, 'arrived_idle', {
                    'lock_state': 'ARRIVED', 'cube_visible': False,
                    'peers_summary': peer_summary, 'slowdown_factor': slowdown_factor,
                    'role': 'ARRIVED',
                    'frames_missing': st['frames_cube_missing_count'],
                    'left': 0.0, 'right': 0.0,
                }

    # 3. STAGE 4 — social attractor. Look for a better-informed peer to follow.
    social_target = find_social_target_peer(bot_id, peers_list, cube_detection)
    if social_target is not None:
        left, right, follow_trace = compute_follow_peer_command(bot_id, social_target, peers_list)

        # If the peer has ARRIVED and we are at following distance, release follower
        # mode so this bot can search for the cube independently from this position.
        if follow_trace['stop_reason'] == 'at_follow_distance' and social_target['peer_is_arrived']:
            print(f"  [{bot_id}] FOLLOWER_RELEASE: {social_target['peer_color']} is ARRIVED — "
                  f"switching to independent search from d={social_target['peer_distance_meters']:.2f}m.")
            # Do NOT return — fall through to section 4 (role-based / search logic).
        else:
            st['last_motor_command_left']  = left
            st['last_motor_command_right'] = right
            if cube_detection is not None:
                st['last_seen_cube_x_normalized'] = cube_detection['cube_x_normalized']
                st['consecutive_missed_frames']   = 0
            state_label = 'following_peer_stopped' if follow_trace['stop_reason'] else 'following_peer'
            return left, right, state_label, {
                'lock_state':       'FOLLOWING',
                'cube_visible':     cube_detection is not None,
                'cube_d':           cube_detection['cube_distance_meters'] if cube_detection else None,
                'cube_x':           cube_detection['cube_x_normalized']    if cube_detection else None,
                'fill_ratio':       compute_frame_fill_ratio(bot_id, cube_detection)
                                      if cube_detection is not None else None,
                'peers_summary':    peer_summary,
                'slowdown_factor':  follow_trace['slowdown_factor'],
                'role':             'FOLLOWER',
                'social_target':    {
                    'color':         social_target['peer_color'],
                    'peer_d':        social_target['peer_distance_meters'],
                    'peer_x':        social_target['peer_x_normalized'],
                    'peer_cube_d':   social_target['peer_cube_distance'],
                    'is_arrived':    social_target['peer_is_arrived'],
                    'reason':        social_target['reason'],
                },
                'forward_target':   follow_trace['forward_target'],
                'turn_intensity':   follow_trace['turn_intensity'],
                'arc_ratio':        follow_trace['arc_ratio'],
                'left': left, 'right': right,
            }

    # 4. No social target → fall through to existing role-based logic.
    current_leader = brain_recompute_and_get_leader()
    is_leader = (current_leader == bot_id)

    # 5. NON-LEADER (no social target, no cube view yet, etc.)
    if not is_leader:
        if cube_detection is not None:
            x = cube_detection['cube_x_normalized']
            d = cube_detection['cube_distance_meters']
            st['last_seen_cube_x_normalized'] = x
            st['consecutive_missed_frames'] = 0
            fill = compute_frame_fill_ratio(bot_id, cube_detection)
            tracker_l, tracker_r = compute_tracker_rotation(bot_id, x, fill)
            st['last_motor_command_left']  = tracker_l
            st['last_motor_command_right'] = tracker_r
            return tracker_l, tracker_r, 'tracking_cube', {
                'lock_state': 'WAITING', 'cube_visible': True,
                'cube_d': d, 'cube_x': x, 'fill_ratio': fill,
                'peers_summary': peer_summary, 'slowdown_factor': slowdown_factor,
                'role': 'TRACKER', 'leader_is': current_leader,
                'left': tracker_l, 'right': tracker_r,
            }
        else:
            st['consecutive_missed_frames'] += 1
            # Stage 5.2: PSO-guided forward arc instead of pure pivot.
            current_x = st['last_seen_cube_x_normalized']
            pso_vx, _ = brain_compute_pso_velocity(bot_id, current_x, 9999.0)
            fwd   = param(bot_id, 'search_forward_speed')
            rot   = param(bot_id, 'lost_rotation_speed')
            max_s = param(bot_id, 'maximum_motor_speed')
            # PSO vx biases arc direction; fall back to last_seen_x when flat.
            if abs(pso_vx) > 0.02:
                turn_dir = math.copysign(1.0, pso_vx)
            elif st['last_seen_cube_x_normalized'] >= 0:
                turn_dir = 1.0
            else:
                turn_dir = -1.0
            left  = min(max_s, fwd + rot * turn_dir)
            right = max(0.0,   fwd - rot * turn_dir)
            rotate_dir = 'right' if turn_dir > 0 else 'left'
            st['last_motor_command_left'], st['last_motor_command_right'] = left, right
            return left, right, f'searching_{rotate_dir}', {
                'lock_state': 'SEARCHING', 'cube_visible': False,
                'last_seen_x': st['last_seen_cube_x_normalized'],
                'rotate_dir': rotate_dir, 'pso_vx': round(pso_vx, 3),
                'peers_summary': peer_summary, 'slowdown_factor': slowdown_factor,
                'role': 'SEARCHER', 'leader_is': current_leader,
                'left': left, 'right': right,
            }

    # 6. LEADER path — approach the cube directly (unchanged from Stage 3.2).
    if cube_detection is not None:
        st['consecutive_missed_frames'] = 0
        d = cube_detection['cube_distance_meters']
        x = cube_detection['cube_x_normalized']
        st['last_seen_cube_x_normalized'] = x
        st['previous_cube_distance'] = d

        fill_ratio = compute_frame_fill_ratio(bot_id, cube_detection)
        if fill_ratio is not None:
            thresh = param(bot_id, 'arrived_cube_fills_frame_ratio')
            if fill_ratio > thresh:
                # Stage 5.1: don't latch ARRIVED if there's a close peer ahead.
                # We want the peer-stop logic to handle approach in that case.
                follow_d = param(bot_id, 'social_target_following_distance')
                nearby_peer_blocking = False
                for p in peers_list:
                    pd = float(p.get('peer_distance_meters', 999.0))
                    px = float(p.get('peer_x_normalized', 0.0))
                    if pd < follow_d + 0.10 and abs(px) < 0.35:
                        nearby_peer_blocking = True
                        break
                if not nearby_peer_blocking:
                    st['arrived_by_fill_latched'] = True
                    st['frames_cube_departed_count'] = 0
                    st['frames_cube_missing_count']  = 0
                    brain_mark_arrived(bot_id)
                    print(f"  [{bot_id}] ARRIVED (fill={fill_ratio:.2f})")
                    st['last_motor_command_left']  = 0.0
                    st['last_motor_command_right'] = 0.0
                    return 0.0, 0.0, 'arrived_by_fill', {
                        'lock_state': 'ARRIVED', 'cube_visible': True,
                        'cube_d': d, 'cube_x': x, 'fill_ratio': fill_ratio,
                        'peers_summary': peer_summary, 'slowdown_factor': slowdown_factor,
                        'role': 'ARRIVED',
                        'left': 0.0, 'right': 0.0,
                    }
                # else: peer blocking arrival; fall through to normal approach
                # logic, which will be clamped by peer repulsion and critical-stop.

        stop_dist  = param(bot_id, 'target_stopping_distance_meters')
        zone       = param(bot_id, 'approach_complete_zone_meters')
        bear_tol   = param(bot_id, 'approach_bearing_tolerance')
        bear_tol_f = param(bot_id, 'approach_bearing_tolerance_final')
        close_fill = param(bot_id, 'close_range_fill_ratio')
        close_bear = param(bot_id, 'close_range_bearing_tolerance')
        creep      = param(bot_id, 'forward_creep_when_bearing_off')
        distance_error = d - stop_dist
        distance_in_zone = abs(distance_error) < zone

        if fill_ratio is not None and fill_ratio > close_fill:
            active_bear_tol = close_bear
        elif distance_in_zone:
            active_bear_tol = bear_tol_f
        else:
            active_bear_tol = bear_tol
        bearing_centered = abs(x) < active_bear_tol

        if distance_in_zone and bearing_centered:
            forward_target = 0.0
            sub_state = 'arrived'
        elif distance_in_zone and not bearing_centered:
            forward_target = creep
            sub_state = 'centering'
        else:
            pull_gain  = param(bot_id, 'forward_pull_gain_per_meter')
            stiction   = param(bot_id, 'motor_stiction_floor')
            max_speed  = param(bot_id, 'maximum_motor_speed')
            min_move   = param(bot_id, 'forward_minimum_to_move')
            fp = max(-1.0, min(1.0, pull_gain * distance_error))
            if abs(fp) < min_move:
                forward_target = 0.0
            elif fp > 0:
                forward_target = min(max_speed, stiction + fp * (max_speed - stiction))
            else:
                forward_target = max(-max_speed, -stiction + fp * (max_speed - stiction))
            sub_state = 'tracking'

        if forward_target > 0:
            forward_target = forward_target * slowdown_factor
            stiction = param(bot_id, 'motor_stiction_floor')
            if forward_target > 0 and forward_target < stiction:
                forward_target = stiction

        tp = param(bot_id, 'turn_pull_gain_per_x_normalized') * x
        if abs(tp) < param(bot_id, 'turn_deadband_in_local_x'):
            cube_turn_intensity = 0.0
        else:
            cube_turn_intensity = max(-1.0, min(1.0, tp * param(bot_id, 'turn_to_wheel_diff_scale')))

        cube_local_vx = cube_turn_intensity
        cube_local_vy = forward_target

        combined_vx = cube_local_vx + peer_rep_vx
        combined_vy = cube_local_vy + peer_rep_vy

        new_forward = combined_vy
        new_turn    = combined_vx

        max_speed = param(bot_id, 'maximum_motor_speed')
        new_forward = max(-max_speed, min(max_speed, new_forward))
        new_turn    = max(-1.0, min(1.0, new_turn))

        left, right, slower_ratio = arc_wheel_mapping(bot_id, new_forward, new_turn)
        st['last_motor_command_left']  = left
        st['last_motor_command_right'] = right

        return left, right, sub_state, {
            'lock_state': 'LOCKED', 'cube_visible': True,
            'cube_d': d, 'cube_x': x, 'fill_ratio': fill_ratio,
            'forward_target': new_forward, 'turn_intensity': new_turn,
            'arc_ratio': slower_ratio, 'slowdown_factor': slowdown_factor,
            'peers_summary': peer_summary,
            'role': 'LEADER', 'leader_is': current_leader,
            'active_bear_tol': active_bear_tol,
            'left': left, 'right': right,
        }

    # Leader can't see cube — lock grace / search.
    st['consecutive_missed_frames'] += 1
    grace_max = param(bot_id, 'cube_lock_grace_frames')

    if st['consecutive_missed_frames'] <= grace_max:
        last_l = st['last_motor_command_left']
        last_r = st['last_motor_command_right']
        approx_forward = (last_l + last_r) / 2.0
        approx_turn    = (last_l - last_r) / 2.0
        if approx_forward > 0:
            approx_forward *= slowdown_factor
        new_forward = approx_forward + peer_rep_vy
        new_turn    = approx_turn + peer_rep_vx
        max_speed = param(bot_id, 'maximum_motor_speed')
        new_forward = max(-max_speed, min(max_speed, new_forward))
        new_turn    = max(-1.0, min(1.0, new_turn))
        left, right, _ = arc_wheel_mapping(bot_id, new_forward, new_turn)
        st['last_motor_command_left'], st['last_motor_command_right'] = left, right
        return left, right, 'lock_grace', {
            'lock_state': 'GRACE', 'cube_visible': False,
            'grace_frames_used': st['consecutive_missed_frames'],
            'last_seen_x': st['last_seen_cube_x_normalized'],
            'peers_summary': peer_summary, 'slowdown_factor': slowdown_factor,
            'role': 'LEADER', 'leader_is': current_leader,
            'left': left, 'right': right,
        }

    # Stage 5.2: PSO-guided forward arc instead of pure pivot.
    current_x = st['last_seen_cube_x_normalized']
    pso_vx, _ = brain_compute_pso_velocity(bot_id, current_x, 9999.0)
    fwd   = param(bot_id, 'search_forward_speed')
    rot   = param(bot_id, 'lost_rotation_speed')
    max_s = param(bot_id, 'maximum_motor_speed')
    if abs(pso_vx) > 0.02:
        turn_dir = math.copysign(1.0, pso_vx)
    elif st['last_seen_cube_x_normalized'] >= 0:
        turn_dir = 1.0
    else:
        turn_dir = -1.0
    left  = min(max_s, fwd + rot * turn_dir)
    right = max(0.0,   fwd - rot * turn_dir)
    rotate_dir = 'right' if turn_dir > 0 else 'left'
    st['last_motor_command_left'], st['last_motor_command_right'] = left, right
    return left, right, f'searching_{rotate_dir}', {
        'lock_state': 'LOST', 'cube_visible': False,
        'grace_frames_used': st['consecutive_missed_frames'],
        'last_seen_x': st['last_seen_cube_x_normalized'],
        'rotate_dir': rotate_dir, 'pso_vx': round(pso_vx, 3),
        'peers_summary': peer_summary, 'slowdown_factor': slowdown_factor,
        'role': 'LEADER', 'leader_is': current_leader,
        'left': left, 'right': right,
    }


print('Controller defined (Stage 5.1: stall detection + follower-aware arrival + clean state reset).')


## §6. Tracking loops

One thread per bot. Each thread runs an independent control loop. Threads don't talk to each other.


In [ ]:
tracking_should_run_by_bot_id = {bid: False for bid in known_bot_ids}
loop_status_lock = threading.Lock()
loop_status_by_bot_id = {bid: {} for bid in known_bot_ids}
iteration_count_by_bot_id = {bid: 0 for bid in known_bot_ids}


def run_tracking_loop(bot_id):
    iteration_count_by_bot_id[bot_id] = 0
    period = param(bot_id, 'control_loop_period_seconds')
    log_every = param(bot_id, 'log_every_n_iterations')
    print(f"[{bot_id}] tracking start")
    while tracking_should_run_by_bot_id[bot_id]:
        iter_start = time.time()
        ok, cube, peers, latency, frame_diff = fetch_state(bot_id)
        if not ok:
            time.sleep(period)
            continue
        left, right, state_label, trace = compute_motor_commands(bot_id, cube, peers, frame_diff)
        send_command(bot_id, left, right)
        iteration_count_by_bot_id[bot_id] += 1
        n = iteration_count_by_bot_id[bot_id]
        with loop_status_lock:
            loop_status_by_bot_id[bot_id] = {
                **trace, 'state': state_label,
                'network_latency_ms': latency * 1000.0,
                'iteration': n,
            }
        if log_every and n % log_every == 0:
            peers_s = ','.join(trace.get('peers_summary', [])) or '-'
            role = trace.get('role', '?')
            cube_str = (f"d={trace.get('cube_d', 0):.2f}"
                        if trace.get('cube_visible') else 'cube=NO')
            fill = trace.get('fill_ratio')
            fill_str = f" fill={fill:.2f}" if fill is not None else ''
            print(f"  [{bot_id}] i={n:4d} [{role}] {state_label:<18s} {cube_str}{fill_str} "
                  f"peers=[{peers_s}] → L={left:+.2f} R={right:+.2f}")
        elapsed = time.time() - iter_start
        if elapsed < period:
            time.sleep(period - elapsed)
    stop_bot(bot_id)
    print(f"[{bot_id}] tracking stopped. iterations={iteration_count_by_bot_id[bot_id]}")


def start_tracking_for_bot(bot_id):
    if tracking_should_run_by_bot_id[bot_id]:
        print(f"[{bot_id}] already running.")
        return
    tracking_should_run_by_bot_id[bot_id] = True
    threading.Thread(target=run_tracking_loop, args=(bot_id,), daemon=True).start()


def start_tracking_for_all_bots():
    reset_bot_states_for_new_run()  # Stage 5.1: wipe per-bot controller state
    brain_reset()                   # clean brain state for each new run
    for bid in known_bot_ids:
        start_tracking_for_bot(bid)


def stop_tracking_for_all_bots():
    for bid in known_bot_ids:
        tracking_should_run_by_bot_id[bid] = False
    time.sleep(0.3)
    stop_all_bots()
    print('All bots stopped.')


print('Tracking loops ready (Stage 5.1 — frame-diff passed through, state reset on start).')


## §7. Multi-pane debug view

One pane per bot. Shows `/camera_annotated` (cube + peer boxes drawn by the bot) and a per-bot status line.


In [ ]:
TRANSPARENT_PNG = base64.b64decode(
    'iVBORw0KGgoAAAANSUhEUgAAAAEAAAABCAQAAAC1HAwCAAAAC0lEQVR42mNkAAIAAAoAAv/lxKUAAAAASUVORK5CYII=')

pane_width  = 400
pane_height = 220

debug_image_widget_by_bot_id = {}
debug_status_label_by_bot_id = {}

for b in bot_registry:
    bid = b['bot_id']
    debug_image_widget_by_bot_id[bid] = widgets.Image(
        value=TRANSPARENT_PNG, format='png', width=pane_width, height=pane_height)
    debug_status_label_by_bot_id[bid] = widgets.Label(value='(idle)')

def make_pane(b):
    bid = b['bot_id']
    hex_c = display_color_hex_for_color.get(b['assigned_color'], '#000')
    title = widgets.HTML(
        f"<b>{bid}</b>  <span style='color:{hex_c};font-weight:bold;'>● {b['assigned_color']}</span>  "
        f"<span style='color:#777;font-size:0.85em;'>{b['ip_address']}</span>")
    return widgets.VBox([title, debug_image_widget_by_bot_id[bid],
                          debug_status_label_by_bot_id[bid]])

panes = [make_pane(b) for b in bot_registry]
if len(panes) == 1:
    grid = panes[0]
elif len(panes) == 2:
    grid = widgets.HBox(panes)
else:
    grid = widgets.VBox([widgets.HBox(panes[:2]),
                          widgets.HBox(panes[2:4]) if len(panes) >= 3 else widgets.HBox([])])
display(grid)

debug_view_should_run = True
debug_camera_period = 0.25
debug_label_period  = 0.15

def _load_overlay_font():
    font_paths_to_try = [
        '/System/Library/Fonts/Supplemental/Arial Bold.ttf',
        '/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf',
        '/usr/share/fonts/dejavu/DejaVuSans-Bold.ttf',
        'C:\\\\Windows\\\\Fonts\\\\arialbd.ttf',
    ]
    from PIL import ImageFont
    for p in font_paths_to_try:
        try:
            return ImageFont.truetype(p, size=20)
        except (IOError, OSError):
            continue
    return ImageFont.load_default()

OVERLAY_FONT = _load_overlay_font()

ROLE_COLORS = {
    'LEADER':          ( 80, 220,  80),
    'TRACKER':         (255, 200,  80),
    'ARRIVED_TRACKER': (120, 200, 255),
    'ARRIVED':         ( 80, 160, 255),
    'SEARCHER':        (200, 150, 200),
    'FOLLOWER':        (180, 180, 180),
}

def overlay_distance_text_on_jpeg(jpeg_bytes, bot_status_snapshot):
    if not jpeg_bytes:
        return jpeg_bytes
    try:
        img = PILImage.open(io.BytesIO(jpeg_bytes)).convert('RGB')
    except Exception:
        return jpeg_bytes
    draw = ImageDraw.Draw(img)
    img_w, img_h = img.size

    lines = []
    text_colors = []

    role = bot_status_snapshot.get('role', '?')
    role_color = ROLE_COLORS.get(role, (255, 255, 255))
    lines.append(f"-- {role} --")
    text_colors.append(role_color)

    if bot_status_snapshot.get('cube_visible'):
        d = bot_status_snapshot.get('cube_d')
        if d is not None:
            lines.append(f"CUBE   {d:.2f} m")
            text_colors.append((255, 80, 80))
        x = bot_status_snapshot.get('cube_x')
        if x is not None:
            lines.append(f"bear   {x:+.2f}")
            text_colors.append((255, 200, 200))
        fill = bot_status_snapshot.get('fill_ratio')
        if fill is not None:
            lines.append(f"fill   {fill:.2f}")
            text_colors.append((255, 220, 60))
    else:
        lines.append('cube   NOT VISIBLE')
        text_colors.append((180, 180, 180))

    peer_rgb_for_color = {
        'red':    (255,  80,  80),
        'orange': (255, 140,  40),
        'yellow': (255, 220,  40),
        'blue':   ( 80, 160, 255),
        'purple': (200, 100, 220),
    }
    for peer_str in bot_status_snapshot.get('peers_summary', []):
        try:
            color_part, dist_part = peer_str.split('@')
            short = color_part[:3]
            lines.append(f"{short:<4s}  {dist_part}")
            text_colors.append(peer_rgb_for_color.get(color_part, (255, 255, 255)))
        except (ValueError, AttributeError):
            continue

    slow = bot_status_snapshot.get('slowdown_factor', 1.0)
    if slow < 0.95:
        lines.append(f"slow   {slow:.2f}")
        text_colors.append((255, 160, 80))

    lock = bot_status_snapshot.get('lock_state')
    if lock == 'PEER_HOLD':
        lines.append('[PEER HOLD]')
        text_colors.append((255, 80, 80))

    pad = 6
    line_height = 24
    box_w = 0
    for line in lines:
        try:
            tb = draw.textbbox((0, 0), line, font=OVERLAY_FONT)
            box_w = max(box_w, tb[2] - tb[0])
        except AttributeError:
            box_w = max(box_w, 8 * len(line))
    box_h = line_height * len(lines) + pad
    draw.rectangle([0, 0, box_w + 2 * pad, box_h], fill=(0, 0, 0))

    for idx, (line, rgb) in enumerate(zip(lines, text_colors)):
        draw.text((pad, pad // 2 + idx * line_height), line, fill=rgb, font=OVERLAY_FONT)

    out = io.BytesIO()
    img.save(out, format='JPEG', quality=80)
    return out.getvalue()

def run_debug_camera_loop_for_bot(bot_id):
    record = bot_record_by_id[bot_id]
    session = http_session_by_bot_id[bot_id]
    while debug_view_should_run:
        try:
            r = session.get(urljoin(base_url_for_bot(record), '/camera_annotated'), timeout=1.5)
            if r.status_code == 200 and r.content:
                with loop_status_lock:
                    status_snap = dict(loop_status_by_bot_id.get(bot_id, {}))
                overlaid_bytes = overlay_distance_text_on_jpeg(r.content, status_snap)
                debug_image_widget_by_bot_id[bot_id].format = 'jpeg'
                debug_image_widget_by_bot_id[bot_id].value = overlaid_bytes
        except requests.RequestException:
            pass
        time.sleep(debug_camera_period)

def run_debug_label_loop():
    while debug_view_should_run:
        with loop_status_lock:
            snap = dict(loop_status_by_bot_id)
        brain_snap = brain_get_status_snapshot()
        for bid, st in snap.items():
            if not st:
                debug_status_label_by_bot_id[bid].value = '(idle)'
                continue
            role = st.get('role', '?')
            state = st.get('state', '?')
            cube_s = (f"d={st.get('cube_d', 0):.2f}m" if st.get('cube_visible') else 'cube=NO')
            fill = st.get('fill_ratio')
            fill_s = f" fill={fill:.2f}" if fill is not None else ''
            peers_s = ','.join(st.get('peers_summary', [])) or '-'
            L = st.get('left', 0); R = st.get('right', 0)
            debug_status_label_by_bot_id[bid].value = (
                f"[{role}] {state} | {cube_s}{fill_s} | peers:{peers_s} | "
                f"L={L:+.2f} R={R:+.2f}")
        time.sleep(debug_label_period)

for bid in known_bot_ids:
    threading.Thread(target=run_debug_camera_loop_for_bot, args=(bid,), daemon=True).start()
threading.Thread(target=run_debug_label_loop, daemon=True).start()
print('Debug view started (Stage 5.1 — STALL_RECOVER role colored as warning).')


### §7.1 Stop debug view

In [ ]:
debug_view_should_run = False
print('Debug view stop requested.')


## §8. Start / Stop tracking

**Before §8.1:**
- Place the red cube in a sensible spot.
- Place the bots so they can ROUGHLY see the cube. (They'll find it if they don't.)
- Look at the debug view in §7 — confirm each bot sees the cube (red box).


### §8.1 START tracking

In [ ]:
start_tracking_for_all_bots()


### §8.2 STOP tracking

In [ ]:
stop_tracking_for_all_bots()


## §9. Shutdown

In [ ]:
stop_tracking_for_all_bots()
debug_view_should_run = False
time.sleep(0.3)
for s in http_session_by_bot_id.values():
    s.close()
print('Shutdown complete.')
